In [ ]:
%matplotlib qt
import mne
import numpy as np
import os.path as op
import matplotlib.pyplot as plt
import scipy as sp
from scipy import signal

In [ ]:
def detrend(data, detrend_type='linear', bp=0):
    """
    Remove linear trend along second axis from 2D array-like object `data`.
    """
    axis = -1
    if detrend_type not in ['linear', 'l', 'constant', 'c']:
        raise ValueError("Trend type must be 'linear' or 'constant'.")  
    data = np.asarray(data)
    if detrend_type in ['constant', 'c']:
        data_detrended = data - np.mean(data, axis, keepdims=True)
        return data_detrended
    else:
        dshape = data.shape
        N = dshape[axis]
        bp = np.sort(np.unique(np.r_[0, bp, N]))
        if np.any(bp > N):
            raise ValueError("Breakpoints must be less than length "
                             "of data along given axis.")
        Nreg = len(bp) - 1 
        # Find leastsq fit and remove it for each piece
        newdata = data.T
        for m in range(Nreg):
            Npts = bp[m + 1] - bp[m]
            A = np.ones((Npts, 2))
            A[:, 0] = np.arange(1, Npts+1) / Npts
            sl = slice(bp[m], bp[m+1])
            coef, _, _, _ = np.linalg.lstsq(A, newdata[sl], rcond=None)
            newdata[sl] = newdata[sl] - np.dot(A, coef)
        data_detrended = newdata.T
        
        return data_detrended

In [ ]:
def computePowerSpectrumDensity(X, Fs, to_window=True, to_plot=True):
    '''
    Compute the Fourier power spectrum of the N-channel block `X`
    [nChannels, nSamples] sampled at frequency `Fs`.
    '''
    n = np.size(X, 1)
    
    # Detrend data
    X = detrend(X)
    
    # Windowing
    if to_window:
        w = np.hamming(n)
        X = w * X

    ## PSD using Scipy
    freq = sp.fft.rfftfreq(n, 1/Fs)
    Xhat = sp.fft.rfft(X)
    scale = 1 / (Fs * (w*w).sum())
    psd = np.real(np.conj(Xhat) * Xhat) * scale # power spectral density
    if n % 2:
        psd[..., 1:] *= 2
    else:
        psd[..., 1:-1] *= 2
        
    # Plot power spectrum
    psd_mean = np.mean(psd, axis=0)
    if to_plot:
        plt.figure()
#         plt.plot(freq, psd.T, ':')
        plt.plot(freq, psd_mean, 'k-', lw=2)
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('PSD [V^2/Hz]')
        plt.show()
        
    return psd, freq

In [ ]:
def compareFFTPower(X, Fs, to_window=True, to_plot=True):
    '''
    Compute the Fourier power spectrum of the N-channel block `X`
    [nChannels, nSamples] sampled at frequency `Fs`.
    '''
    n = np.size(X, 1)
    
    # Detrend data
    X = detrend(X)
    
    # Windowing
    if to_window:
        w = np.hamming(n)
        X = w * X
#         plt.figure()
#         plt.plot(np.arange(n), X.T)
#         plt.title('Windowed signal')
#         plt.show()
    
    # FFT using Numpy
    Xhat = np.fft.rfft(X)
    scale = 1 / w.sum()**2
    power = np.real(Xhat * np.conj(Xhat)) * scale # FFT power spectrum
    if n % 2:
        power[..., 1:] *= 2
    else:
        power[..., 1:-1] *= 2
    freq = Fs / n * np.arange(n)
    k = np.arange(1, np.floor(n / 2), dtype='int') # only plot the first half of frequencies
    freq_M0 = freq[k]; power_M0 = power[:, k]; power_M0_mean = np.mean(power_M0, axis=0)

    ## PSD using Scipy - Method-1
    Xhat = sp.fft.rfft(X)
    scale = 1 / w.sum()**2
    power = np.real(Xhat * np.conj(Xhat)) * scale # FFT power spectrum
    if n % 2:
        power[..., 1:] *= 2
    else:
        power[..., 1:-1] *= 2
    freq = Fs / n * np.arange(n)
    k = np.arange(1, np.floor(n / 2), dtype='int') # only plot the first half of frequencies
    freq_M1 = freq[k]; power_M1 = power[:, k]; power_M1_mean = np.mean(power_M1, axis=0)
    
#     Xhat = sp.fft.fft(X)
#     psd = np.real(np.conj(Xhat) * Xhat) / n # power spectral density
#     freq = Fs / n * np.arange(n)
#     k = np.arange(1, np.floor(n / 2), dtype='int') # only plot the first half of frequencies
#     freq = freq[k]; psd = psd[:, k]
    
    ## PSD using Scipy - Method-2
#     freq, psd = signal.periodogram(X, Fs)

    ## PSD using Scipy - Method-3
    freq_M2, power_M2 = signal.welch(X, Fs, nperseg=X.shape[-1], noverlap=0, scaling='spectrum')
    power_M2_mean = np.mean(power_M2, axis=0)

    # Plot power spectrum
    if to_plot:
        plt.figure()
        plt.plot(freq_M0, power_M0_mean, label='Numpy')
        plt.plot(freq_M1, power_M1_mean, label='Scipy')
        plt.plot(freq_M2, power_M2_mean, label='Scipy-Welch')
        plt.legend()
        plt.show()
        
    return power_M0, freq_M0

In [ ]:
# Read preprocessed EEG data
fmax = 170 # Hz
data_path = '~/data/sample'
raw_file = op.join(data_path, 'sample_audvis_eeg.fif')
raw = mne.io.read_raw_fif(raw_file)
raw.crop(tmin=216, tmax=276) # pseudo-resting data interval

In [ ]:
# Make equally spaced events for resting state epochs
events = mne.make_fixed_length_events(raw, duration=1.2)

# Segment resting state data into epochs
epochs = mne.Epochs(raw, events, tmax=1, preload=True)

In [ ]:
X = np.squeeze(epochs[0].get_data(picks='eeg'))
print('Shape of X =', np.shape(X))

In [ ]:
psd_M0, freqs_M0 = compareFFTPower(X, raw.info['sfreq'], to_window=True, to_plot=True)
# psd_M0, freqs_M0 = computePowerSpectrumDensity(X, raw.info['sfreq'],
#                                                  to_window=True, to_plot=True)
# psd_M0_mean = np.mean(psd_M0, axis=0)

# plt.figure()
# plt.plot(freqs_M0, psd_M0_mean, '-')
# plt.xlim([0, 170])
# plt.show()

In [ ]:
n_channels = np.size(X, 0)
new_info = mne.create_info(n_channels, sfreq=raw.info['sfreq'])

epoch_X = mne.EpochsArray(np.expand_dims(X, 0), new_info)
np.shape(epoch_X)

In [ ]:
psds_MNE, freqs_MNE = mne.time_frequency.psd_welch(epoch_X, average='mean',
                                                   n_fft=np.size(epoch_X, 2), picks='all')

plt.figure()
plt.plot(freqs_MNE, psds_MNE.mean(axis=1).T, '-')
plt.xlim([0, 170])
plt.show()

In [ ]:
722 % 2